In [15]:
import torch 
import torch.nn as nn
from einops import rearrange 
from mamba_ssm import Mamba
import math

## Implementing Visual Mamba Block

In [2]:
class VisionMambaBlock(nn.Module):
    """
    A Wrapper around the core Mamba block to handle 2D Images.
    Includes Bidirectional scanning (Forward + Backward) to simulate 
    global spatial contet (non-causal)
    """
    def __init__(self, dim, d_state = 16, d_conv = 4, expand = 2, bimamba = True):
        super().__init__()
        self.bimamba = bimamba 

        # 1. Forward Scan
        # Note: If mamba_ssm is missing, these will fail. ensure you have the fallback logic.
        self.mamba_fwd = Mamba(
            d_model = dim,
            d_state = d_state, 
            d_conv = d_conv, 
            expand = expand
        )
        if bimamba:
            # 2. Backward Scan
            self.mamba_bwd = Mamba(
                d_model = dim,
                d_state = d_state, 
                d_conv = d_conv, 
                expand = expand
            )

            # 3. The Fusion Layer (Concatenation -> Project back to dim)
            # We take 2 * dim inputs and project back to dim
            self.fusion_linear = nn.Linear(dim * 2, dim)

        self.norm = nn.LayerNorm(dim)

        # Skip connection linear projection (optional, helps convergence)
        self.skip_scale = nn.Parameters(torch.ones(dim))

    def forward(self, x):
        """
        Input: (B, C, H, W) -> Standard PyTorch Image format
        Output: (B, C, H, W)
        """

        B, C, H, W = x.shape 
        assert C == self.dim, f"Input channels {C} must match the Mamba dim {self.dim}"

        # 1. Flatten (B, C, H, W) -> (B, H * W, C)
        x_flat = rearrange(x,  'b c h w -> b (h w) c')

        # 2. Apply Norm 
        x_norm = self.norm(x_flat)

        # 3. Mamba Forward Pass (Top-left to Bottom Right)
        if self.bimamba:
            # 4. Mamba Backward Pass (Bottom-Right to Top-Left)
            # Flip the sequence along the length dimension
            x_flip = torch.flip(x_norm, dims = [1])
            out_bwd = self.mamba_bwd(x_flip)
            
            # FLip back to original order 
            out_bwd = torch.flip(out_bwd, dims=[1])

            # Combine: simple average or addition 
            combined = torch.cat([out_fwd, out_bwd], dim = -1)
            out = self.fusion_linear(combined)
        else:
            out = out_fwd

        # 5. Residual Connection + Reshape back to Image
        out = out * self.skip_scale # Scale stability
        out = out + x_flat # Residual with the flat input
        
        # (B, H*W, C) -> (B, C, H, W)
        out = rearrange(out, 'b (h w) c -> b c h w', h=H, w=W)
            
        return out

The Missing Ingredients for Dehazing

Mamba is amazing at Spatial Mixing (pixel $i$ talks to pixel $j$), but it is weak at Channel Mixing (Blue talks to Red). Dehazing requires fixing color casts (Atmospheric light), which is a Channel problem.

In [3]:
class ChannelAttention(nn.Module):
    """Simple Squeeze-and-Excitation block for Color Correction"""
    def __init__(self, dim, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(dim, dim // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(dim // reduction, dim, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

In [4]:
class SinusoidalPosEmb(nn.Module):
    """
    Note that the implementation is a little bit different from the
    Transformers paper but when fed in the linear layers, they learn the weighted
    sums accross the entire input vector
    => All positional information are fully encoded
    """

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2

        exponential_denominator = half_dim - 1
        log_base = math.log(10000.0)
        c = log_base / (exponential_denominator)

        # Frequencies = 1 / (10000 ^ (i / (d_model // 2 - 1))
        frequencies = torch.exp(torch.arange(half_dim, device=device) * -c)

        # Apply the frequencies to the input scaler (t)
        # t has shape (batch_size, 1) and frequencies has shape (1, half_dim)
        arguments = x[:, None] * frequencies[None, :]

        return torch.cat((arguments.sin(), arguments.cos()), dim=-1)

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2

        exponential_denominator = half_dim - 1
        log_base = math.log(10000.0)
        c = log_base / (exponential_denominator)

        # Frequencies = 1 / (10000 ^ (i / (d_model // 2 - 1))
        frequencies = torch.exp(torch.arange(half_dim, device=device) * -c)

        arguments = x[:, None] * frequencies[None, :]

        sin_component = arguments.sin()
        cos_component = arguments.cos()

        stacked = torch.stack((sin_component, cos_component), dim=-1)

        # Reshape(batch_size, half_dim, 2) to (batch_size, half_dim * 2)
        # Collapsing the final dimension and let those elements interleaved together
        interleaved_eb = stacked.view(x.shape[0], -1)
        return interleaved_eb


### Upblock for Unet

In [5]:
class ResNetBlock(nn.Module):
    def __init__(self, dim, dim_out, time_emb_dim=None, groups=8, use_checkpoint=False):
        super().__init__()
        self.use_checkpoint = use_checkpoint

        self.time_mlp = (
            nn.Sequential(nn.Linear(time_emb_dim, dim_out), nn.SiLU())
            if time_emb_dim
            else None
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(dim, dim_out, kernel_size=3, padding=1),
            nn.GroupNorm(groups, dim_out),
            nn.SiLU(),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(dim_out, dim_out, kernel_size=3, padding=1),
            nn.GroupNorm(groups, dim_out),
            nn.SiLU(),
        )
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()

    def forward_impl(self, x, time_embed=None):
        h = self.block1(x)
        if self.time_mlp is not None and time_embed is not None:
            h = h + self.time_mlp(time_embed)[:, :, None, None]
        h = self.block2(h)
        shortcut_h = self.res_conv(x)

        return h + shortcut_h

    def forward(self, x, time_embed=None):
        if self.training and self.use_checkpoint:
            # Checkpointing requires inputs to require_grad for backward to work correctly
            # Often x requires grad, but time_embed might not.
            return checkpoint.checkpoint(
                self.forward_impl, x, time_embed, use_reentrant=False
            )
        else:
            return self.forward_impl(x, time_embed)


In [6]:
class MultiScaleVisionMamba(nn.Module):
    def __init__(self, dim, d_state = 16, d_conv=4, expand=2, bimamba=True, groups=8):
        super().__init__()

        # --- PART 1: Your Original Multi-Scale Context (MSC) ---
        # Keep this! It extracts features at different dilation rates.
        internal_dim = dim // 4
        self.msc_conv1 = nn.Conv2d(dim, internal_dim, kernel_size=3, padding=1, dilation=1)
        self.msc_conv2 = nn.Conv2d(dim, internal_dim, kernel_size=3, padding=2, dilation=2)
        self.msc_conv3 = nn.Conv2d(dim, internal_dim, kernel_size=3, padding=4, dilation=4)
        self.msc_conv4 = nn.Conv2d(dim, dim - (3 * internal_dim), kernel_size=3, padding=1, dilation=1)
        
        self.msc_merge = nn.Sequential(
            nn.Conv2d(dim * 2, dim, kernel_size=1), 
            nn.GroupNorm(groups, dim), 
            nn.SiLU()
        )

        # 2. Channel Attention (Color Correction - Critical for Dehazing)
        self.channel_attn = ChannelAttention(dim)

        # --- PART 3: Mamba (Replaces the Transformer Attention) ---
        # Instead of Patch Embedding + QKV + Softmax, we use SSM.
        self.dim = dim
        self.bimamba = bimamba
        self.norm = nn.LayerNorm(dim)
        
        # Mamba Forward
        self.mamba_fwd = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
        if bimamba:
            # Mamba Backward
            self.mamba_bwd = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
            self.fusion_linear = nn.Linear(dim * 2, dim)

        # 4. Gating Parameter (Learnable weight)
        # Allows the model to say "I don't need Mamba for this simple pixel"
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x shape: (B, C, H, W)
        shortcut = x

        # 1. Run Multi-Scale Context (Your original logic)
        x1 = self.msc_conv1(x)
        x2 = self.msc_conv2(x)
        x3 = self.msc_conv3(x)
        x4 = self.msc_conv4(x)
        x_msc = torch.cat([x1, x2, x3, x4], dim=1)

        # Merge input with MSC features
        x_local = self.msc_merge(torch.cat([x, x_msc], dim=1))

        # 2. Channel Attention (Fix Color Casts)
        x_local = self.channel_attn(x_local)

        # 2. Prepare for Mamba (Flatten spatial dims)
        B, C, H, W = x_local.shape
        # (B, C, H, W) -> (B, H*W, C)
        x_flat = rearrange(x_local, 'b c h w -> b (h w) c')
        x_norm = self.norm(x_flat)

        # 3. Bidirectional Mamba Scan
        out_fwd = self.mamba_fwd(x_norm)
        
        if self.bimamba:
            x_flip = torch.flip(x_norm, dims=[1])
            out_bwd = self.mamba_bwd(x_flip)
            out_bwd = torch.flip(out_bwd, dims=[1])
            
            # Fuse
            combined = torch.cat([out_fwd, out_bwd], dim=-1)
            x_global = self.fusion_linear(combined)
        else:
            x_global = out_fwd

        # 4. Final Projection & Reshape
        x_global = rearrange(x_global, 'b (h w) c -> b c h w', h=H, w=W)

        # --- D. Gated Fusion ---
        # Output = Original + Local + (gamma * Global)
        # Initializing gamma at 0 makes training very stable (starts as pure CNN)
        return shortcut + x_local + (self.gamma * x_global)

In [7]:
class DownBlock(nn.Module):
    def __init__(
        self,
        dim_in,
        dim_out,
        attn=False,
        time_embed_dim=256,
        groups=8,
        use_checkpoint=False,
        d_state=16,    # Latent state (higher = more memory of history)
        d_conv=4,      # Local conv kernel size (higher = smoother local context)
        expand=2,      # Expansion factor (higher = more powerful mixing)
        bimamba=True   # True = scan forward and backward (Better for images)
    ):
        super().__init__()
        
        # 1. ResNet Block 1
        self.res_block1 = ResNetBlock(
            dim_in,
            dim_out,
            time_emb_dim=time_embed_dim,
            groups=groups,
            use_checkpoint=use_checkpoint,
        )
        
        # 2. ResNet Block 2
        self.res_block2 = ResNetBlock(
            dim_out,
            dim_out,
            time_emb_dim=time_embed_dim,
            groups=groups,
            use_checkpoint=use_checkpoint,
        )
        
        # 3. The Context Block (Mamba)
        # We now pass the custom parameters into the block
        if attn:
            self.context_block = MultiScaleVisionMamba(
                dim=dim_out,      # Input channels
                d_state=d_state,  # Custom Param
                d_conv=d_conv,    # Custom Param
                expand=expand,    # Custom Param
                bimamba=bimamba,  # Custom Param
                groups=groups
            )
        else:
            self.context_block = nn.Identity()

        # 4. Downsample
        self.downsample = nn.Conv2d(
            dim_out, dim_out, kernel_size=4, stride=2, padding=1
        )

    def forward(self, x, t_emb):
        x = self.res_block1(x, t_emb)
        
        # Apply Mamba (or Identity)
        x = self.context_block(x)
        
        x = self.res_block2(x, t_emb)

        skip_feature = x
        x = self.downsample(x)

        return x, skip_feature

In [8]:
class UpBlock(nn.Module):
    def __init__(
        self,
        dim_in,
        dim_skip,
        dim_out,
        attn=False,          # Controls if Mamba is used
        time_embed_dim=256,
        groups=8,
        use_checkpoint=False,
        d_state=16,
        d_conv=4,
        expand=2,
        bimamba=True
    ):
        super().__init__()
        
        # 1. Upsampling (Double the size)
        self.upsample = nn.Upsample(
            scale_factor=2, mode="bilinear", align_corners=False
        )
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, padding=1)

        # 2. ResBlock 1: Merges the "Up" features with "Skip" features
        # Input channels = dim_out (from upsample) + dim_skip (from encoder)
        self.res_block1 = ResNetBlock(
            dim_out + dim_skip,
            dim_out,
            time_emb_dim=time_embed_dim,
            groups=groups,
            use_checkpoint=use_checkpoint,
        )

        # 3. The Context Block (MultiScale Vision Mamba)
        # Replaces the old Attention logic
        if attn:
            self.context_block = MultiScaleVisionMamba(
                dim=dim_out,      # Operates on the merged dim_out
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                bimamba=bimamba,
                groups=groups
            )
        else:
            self.context_block = nn.Identity()

        # 4. ResBlock 2: Refines features after Mamba
        self.res_block2 = ResNetBlock(
            dim_out,
            dim_out,
            time_emb_dim=time_embed_dim,
            groups=groups,
            use_checkpoint=use_checkpoint,
        )

    def forward(self, x, time_embed, skip):
        # 1. Upsample
        x = self.upsample(x)
        x = self.conv(x)
        
        # 2. Concatenate with Skip Connection
        # x: (B, dim_out, H, W)
        # skip: (B, dim_skip, H, W)
        x = torch.cat([x, skip], dim=1) 
        
        # 3. Process Merged Features (Reduce channels back to dim_out)
        x = self.res_block1(x, time_embed)
        
        # 4. Global Context (Mamba)
        x = self.context_block(x)
        
        # 5. Final Refinement
        x = self.res_block2(x, time_embed)

        return x

### UNet Model

In [17]:
class PhysMambaUNet(nn.Module):
    def __init__(
        self,
        dim=64,                # Base channel dimension
        init_dim=None,         # Initial convolution dimension
        out_dim=3,             # Output channels (3 for RGB dehazing)
        dim_mults=(1, 2, 4, 8),# Channel multipliers per level
        channels=3,            # Input channels
        with_time_emb=True,    # Whether to use time embedding (for Diffusion)
        resnet_block_groups=8,
        # --- Mamba Config ---
        use_mamba_encoder=False, # Save memory: Don't use Mamba in early down-blocks
        use_mamba_bottleneck=True,
        use_mamba_decoder=True,  # Use Mamba in up-blocks for reconstruction
        d_state=16,
        d_conv=4, 
        expand = 2
    ):
        super().__init__()

        # 1. Dimensions setup
        self.channels = channels
        init_dim = default(init_dim, dim)
        self.init_conv = nn.Conv2d(channels, init_dim, 7, padding=3)
        
        dims = [init_dim, *map(lambda m: dim * m, dim_mults)]
        in_out = list(zip(dims[:-1], dims[1:])) # [(64, 128), (128, 256), ...]

        # 2. Time Embeddings (Standard for Diffusion)
        if with_time_emb:
            time_dim = dim * 4
            self.time_mlp = nn.Sequential(
                SinusoidalPosEmb(dim),
                nn.Linear(dim, time_dim),
                nn.GELU(),
                nn.Linear(time_dim, time_dim),
            )
        else:
            time_dim = None
            self.time_mlp = None

        # 3. Encoder (DownBlocks)
        self.downs = nn.ModuleList([])
        num_resolutions = len(in_out)

        for ind, (dim_in, dim_out) in enumerate(in_out):
            is_last = ind >= (num_resolutions - 1)
            
            # Strategy: Only use Mamba in deep layers or if explicitly requested
            use_attn = use_mamba_encoder and (ind >= 2) 

            self.downs.append(
                DownBlock(
                    dim_in,
                    dim_out,
                    attn=use_attn, # Toggle Mamba
                    time_embed_dim=time_dim,
                    groups=resnet_block_groups,
                    d_state=d_state,
                    d_conv=d_conv,
                    expand=expand
                )
            )

        # 4. Bottleneck (The Core "Global Physics" Solver)
        mid_dim = dims[-1]
        self.mid_block1 = ResNetBlock(mid_dim, mid_dim, time_emb_dim=time_dim, groups=resnet_block_groups)
        
        if use_mamba_bottleneck:
            self.mid_attn = MultiScaleVisionMamba(
                mid_dim, 
                d_state=64,       # Higher state for bottleneck (Critical!)
                d_conv=d_conv, 
                expand=expand, 
                bimamba=True
            )
        else:
            self.mid_attn = nn.Identity()
            
        self.mid_block2 = ResNetBlock(mid_dim, mid_dim, time_emb_dim=time_dim, groups=resnet_block_groups)

        # 5. Decoder (UpBlocks)
        self.ups = nn.ModuleList([])
        
        # We iterate backwards through dimensions
        for ind, (dim_in, dim_out) in enumerate(reversed(in_out)):
            is_last = ind == (len(in_out) - 1)
            
            # Strategy: Use Mamba in decoder to "hallucinate" clean details
            use_attn = use_mamba_decoder

            self.ups.append(
                UpBlock(
                    dim_in = dim_out,    # Input from previous UpBlock
                    dim_skip = dim_out,     # Skip connection dimension
                    dim_out = dim_in,     # Output dimension (target)
                    attn=use_attn,
                    time_embed_dim=time_dim,
                    groups=resnet_block_groups,
                    d_state=d_state,
                    d_conv=d_conv,
                    expand=expand
                )
            )

        # 6. Final Projection
        self.final_res_block = ResNetBlock(init_dim * 2, init_dim, time_emb_dim=time_dim, groups=resnet_block_groups)
        self.final_conv = nn.Conv2d(init_dim, out_dim, 1)

    def forward(self, x, time=None):
        # 1. Time Embedding
        t = None
        if self.time_mlp is not None:
            if time is None:
                raise ValueError("Time step must be provided for diffusion models")
            t = self.time_mlp(time)

        # 2. Initial Convolution
        x = self.init_conv(x)
        r = x.clone() # Keep residual for end

        # 3. Encoder Scan (Down)
        h = [] # Store skip connections
        for block in self.downs:
            x, skip = block(x, t)
            h.append(skip)

        # 4. Bottleneck Scan (Middle)
        x = self.mid_block1(x, t)
        x = self.mid_attn(x) # <--- Heavy Mamba Global Context
        x = self.mid_block2(x, t)

        # 5. Decoder Scan (Up)
        for block in self.ups:
            skip = h.pop()
            x = block(x, t, skip)

        # 6. Final Block
        x = torch.cat((x, r), dim=1) # Concat with original residual
        x = self.final_res_block(x, t)
        
        return self.final_conv(x)

def default(val, d):
    if val is not None:
        return val
    return d

In [18]:
def get_phys_mamba_model(version='medium', device='cuda'):
    """
    Factory function to create Small, Medium, or Large variants.
    """
    if version == 'small':
        # --- SMALL (Mobile/Edge) ---
        # Lightweight. Mamba only in the bottleneck.
        # fast inference, low VRAM.
        model = PhysMambaUNet(
            dim=32,
            dim_mults=(1, 2, 4),        # Shallower (3 levels)
            use_mamba_encoder=False,    # Pure CNN Encoder
            use_mamba_bottleneck=True,  # Mamba only here
            use_mamba_decoder=False,    # Pure CNN Decoder
            d_state=16,
            d_conv=2,                   # Smaller kernel
            expand=1.5                  # Lower expansion to save params
        )

    elif version == 'medium':
        # --- MEDIUM (The Standard) ---
        # Balanced. Mamba in Bottleneck and Decoder (for reconstruction).
        # Good for RTX 3060/4060 class cards.
        model = PhysMambaUNet(
            dim=64,
            dim_mults=(1, 2, 4, 8),     # Standard Depth (4 levels)
            use_mamba_encoder=False,    # Keep encoder fast
            use_mamba_bottleneck=True,
            use_mamba_decoder=True,     # Help reconstruct clean details
            d_state=16,
            d_conv=4,
            expand=2
        )

    elif version == 'large':
        # --- LARGE (SOTA Chaser) ---
        # Heavy. Mamba EVERYWHERE. 
        # Needs A100 or 4090 (24GB VRAM recommended for training).
        model = PhysMambaUNet(
            dim=96,                     # Wider channels
            dim_mults=(1, 2, 4, 8),
            use_mamba_encoder=True,     # Full Global Context
            use_mamba_bottleneck=True,
            use_mamba_decoder=True,
            d_state=32,                 # Higher state capacity
            d_conv=4,
            expand=2
        )
    
    else:
        raise ValueError("Version must be small, medium, or large")

    return model.to(device)

# Profiling Model

In [21]:
import torch
import torch.nn as nn
from thop import profile, clever_format

# Import your model classes here
# from model import get_phys_mamba_model

def profile_model(version, input_size=(1, 3, 256, 256)):
    print(f"\n--- Profiling {version.upper()} Model ---")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Instantiate
    model = get_phys_mamba_model(version, device=device)
    # print(model)
    model.eval()
    
    # 2. Create Dummy Input
    input_tensor = torch.randn(input_size).to(device)
    # If using diffusion, we need time embedding too
    t = torch.randn(1).to(device) 

    # 3. Calculate FLOPs and Params using THOP
    # Note: We wrap the forward pass to handle the 'time' argument if needed
    macs, params = profile(model, inputs=(input_tensor, t), verbose=False)
    
    # 4. Measure Memory (Peak VRAM during forward pass)
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model(input_tensor, t)
    max_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB

    # 5. Format Output
    macs_fmt, params_fmt = clever_format([macs, params], "%.3f")
    
    print(f"Parameters: {params_fmt}")
    print(f"FLOPs (MACs): {macs_fmt}")
    print(f"VRAM (Inference): {max_mem:.2f} MB")
    
    return params, macs

if __name__ == "__main__":
    # Profile all three
    profile_model('small')
    profile_model('medium')
    profile_model('large')


--- Profiling SMALL Model ---
Parameters: 2.574M
FLOPs (MACs): 17.019G
VRAM (Inference): 746.85 MB

--- Profiling MEDIUM Model ---
Parameters: 40.622M
FLOPs (MACs): 91.007G
VRAM (Inference): 1205.62 MB

--- Profiling LARGE Model ---
Parameters: 104.439M
FLOPs (MACs): 226.152G
VRAM (Inference): 1680.93 MB
